In [ ]:
import { LogicParser, isTerm } from "./FOL-Parser";

# <a href="https://en.wikipedia.org/wiki/Unification_(computer_science)">Unification</a>

This notebook implements the algorithm of *Martelli and Montanari* for the unification of terms.

## Utility Functions & Strict Typing

In [ ]:
type Variable = string;
type Term = Variable | [string, ...Term[]];
type Equation = ['≐', Term, Term];
type Substitution = Map<Variable, Term>;

Formulas are represented as nested tuples.  In order to convert a string into a nested tuple we use the class `LogicParser` that is implemented in the module `folParser`.  Our parser distinguishes variables and function symbol as follows:
- A word starting with a lower case letter is interpreted as a *variable*.
- A word starting with an upper case letter is assumed to be a *function* or *predicate symbol*.

In [ ]:
function parseTerm(s: string): Term {
    const ast = new LogicParser(s).parse();
    if (isTerm(ast)) return ast as Term;
    throw new Error(`Parsed AST is not a valid Term: ${JSON.stringify(ast)}`);
}

In [ ]:
console.dir(parseTerm('f(g(X),Y)'), { depth: null });

The method $\texttt{apply}(t, \sigma)$ takes an object $t$ and a substitution $\sigma$ and computes the $t\sigma$, i.e. it *applies* the substitution $\sigma$ to $t$.  The object $t$ is either a term, a *syntactic equation*, or a set of syntactic equations.  The substitution $\sigma$ is represented as a dictionary.  Assume that $\sigma = \bigl\{ x_1 \mapsto t_1, \cdots, x_n \mapsto t_n \bigr\}$.  Then $t\sigma$ is defined by induction on $t$:
- If $t$ is a variable, there are two cases when defining $t\sigma$:
  - $t = x_i$ for an $i\in\{1,\cdots,n\}$.  Then we define  
    $$ x_i\sigma := t_i. $$
  - $t = y$ where $y\in\mathcal{V}$, but $y \not\in \{x_1,\cdots,x_n\}$. Then we define   
    $$ y\sigma := y.$$</li>
- Otherwise, we must have $t = f(s_1,\cdots,s_m)$. Then we define: 
  $$ f(s_1, \cdots, s_m)\sigma := f(s_1\sigma, \cdots, s_m\sigma). $$

In [ ]:
// We separate apply into typed variants to avoid monolithic 'any' processing
function applyTerm(t: Term, sigma: Substitution): Term {
    if (typeof t == 'string') {
        const mapped = sigma.get(t);
        return mapped !== undefined ? mapped : t;
    }
    const [f, ...args] = t;
    return [f, ...args.map(arg => applyTerm(arg, sigma))];
}

In [ ]:
function applyEquation(eq: Equation, sigma: Substitution): Equation {
    const [op, left, right] = eq;
    return [op, applyTerm(left, sigma), applyTerm(right, sigma)];
}

In [ ]:
function applyEquations(E: Equation[], sigma: Substitution): Equation[] {
    return E.map(eq => applyEquation(eq, sigma));
}

In [ ]:
const s1 = parseTerm('g(Z)');
const s2 = parseTerm('h(U, V)');

const sigma: Substitution = new Map([
    ['X', s1],
    ['Y', s2]
]);
console.log(sigma);

In [ ]:
const tTerm = parseTerm('f(X,h(Y,X),g(Z))');
console.dir(tTerm, { depth: null });

In [ ]:
console.dir(applyTerm(tTerm, sigma), { depth: null });

If  $\sigma = \big\{ x_1 \mapsto s_1, \cdots, x_m \mapsto s_m \big\}$ and
$\tau = \big\{ y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$ 
are two substitutions that are <em style="color:blue;">non-overlapping</em>, i.e. such that $\texttt{dom}(\sigma) \cap \texttt{dom}(\tau) = \{\}$ holds,
then we define the <em style="color:blue;">composition</em> $\sigma\tau$ of $\sigma$ and $\tau$ as follows:
$$\sigma\tau := \big\{ x_1 \mapsto s_1\tau, \cdots, x_m \mapsto s_m\tau,\; y_1 \mapsto t_1, \cdots, y_n \mapsto t_n \big\}$$
The function $\texttt{compose}(\sigma, \tau)$ takes two non-overlapping substitutions and computes the composition $\sigma\tau$.

In [ ]:
function compose(sigma: Substitution, tau: Substitution): Substitution {
    const appliedSigma = [...sigma.entries()].map(
        ([x, s]): [Variable, Term] => [x, applyTerm(s, tau)]
    );
    return new Map([...appliedSigma, ...tau]);
}

In [ ]:
const tau: Substitution = new Map([
    ['Z', s1], 
    ['U', s2]
]);

console.log("Sigma:", sigma);
console.log("Tau:", tau);
console.log("Composed:", compose(sigma, tau));

The function $\texttt{occurs}(x, t)$ checks whether the variable $x$ occurs in the term $t$.

In [ ]:
function occurs(x: Variable, t: Term): boolean {
    if (x == t)               { return true; }
    if (typeof t == 'string') { return false; }
    const [_, ...args] = t;
    return args.some(arg => occurs(x, arg));
}

In [ ]:
console.dir(tTerm, { depth: null });
console.log("Occurs 'U':", occurs('U', tTerm));
console.log("Occurs 'X':", occurs('X', tTerm));

## The Algorithm of Martelli and Montanari

The rules of Martelli and Montanari that can be used to solve a system of syntactical equations are as follows:
<ol>
<li> If $y\in\mathcal{V}$ is a variable that does <b style="color:red;">not</b> occur in the term $t$,
     then we perform the following reduction: 
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle \quad\leadsto \quad 
         \Big\langle E\{y \mapsto t\}, \sigma\big\{ y \mapsto t \big\} \Big\rangle 
     $$
</li>      
<li> If the variable $y$ occurs in the term $t$, then the system of syntactical equations
     $E \cup \big\{ y \doteq t \big\}$ is not solvable:
     $$ \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle\;\leadsto\; \texttt{None} \quad
        \mbox{if $y \in \textrm{Var}(t)$ and $y \not=t$.}$$
</li>
<li> If $y\in\mathcal{V}$ is a variable and $t$ is no variable, then we use the following rule:
     $$ \Big\langle E \cup \big\{ t \doteq y \big\}, \sigma \Big\rangle \quad\leadsto \quad 
         \Big\langle E \cup \big\{ y \doteq t \big\}, \sigma \Big\rangle.
     $$   
</li>
<li> Trivial syntactical equations of variables can be dropped:
     $$ \Big\langle E \cup \big\{ x \doteq x \big\}, \sigma \Big\rangle \quad\leadsto \quad
         \Big\langle E, \sigma \Big\rangle.
     $$   
</li>
<li> If $f$ is an $n$-ary function symbol, then we have: 
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_n) \doteq f(t_1,\cdots,t_n) \big\}, \sigma \Big\rangle 
         \;\leadsto\; 
         \Big\langle E \cup \big\{ s_1 \doteq t_1, \cdots, s_n \doteq t_n\}, \sigma \Big\rangle.
     $$   
</li>
<li> The system of syntactical equations $E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\}$
     has <b style="color:red;">no</b> solution if the function symbols $f$ and $g$ are different:
     $$ \Big\langle E \cup \big\{ f(s_1,\cdots,s_m) \doteq g(t_1,\cdots,t_n) \big\},
      \sigma \Big\rangle \;\leadsto\; \texttt{None} \qquad \mbox{if $f \not= g$}.
     $$
</ol>


Given two terms $s$ and $t$, the function $\texttt{unify}(s, t)$ computes the <em style="color:blue;">most general unifier</em> of $s$ and $t$.

In [ ]:
function solve(E: Equation[], sigma: Substitution): Substitution | null {
    if (E.length == 0) { return sigma; }
    const [eq, ...restE] = E;
    const [_, s, t] = eq;
    if (s == t) {
        return solve(restE, sigma);
    }
    if (typeof s == 'string') {
        if (occurs(s, t)) { 
            return null;
        }
        const subS: Substitution = new Map([[s, t]]);
        return solve(applyEquations(restE, subS), compose(sigma, subS));
    }
    if (typeof t == 'string') {
        const flippedEq: Equation = ['≐', t, s];
        return solve([flippedEq, ...restE], sigma);
    }
    const [f, ...sArgs] = s;
    const [g, ...tArgs] = t;   
    if (f != g || sArgs.length != tArgs.length) {
        return null;
    }
    const newEqs = sArgs.map((sArg, i): Equation => ['≐', sArg, tArgs[i]]);
    return solve([...newEqs, ...restE], sigma);
}

In [ ]:
function unify(s: Term, t: Term): Substitution | null {
    return solve([['≐', s, t]], new Map());
}

Given a list of <em style="color:blue;">syntactical equations</em> $E$ and a substitution $\sigma$, the function $\texttt{solve}(E, \sigma)$ uses the rules of Martelli and Montanari to solve $E$ recursively.

In [ ]:
const t1 = parseTerm('p(X1,f(X1))');
const t2 = parseTerm('p(g(X2),X3)');
console.dir([t1, t2], { depth: null });

In [ ]:
const mu = unify(t1, t2);
console.log(mu);